# 07 — CSE-CIC-IDS2018: download and audit

Replicates the Sect. 4 audit on the second CIC benchmark.

**Design constraint, decided up front:** the official 2018 ML CSVs drop
Flow ID / IPs / ports for **9 of 10 days**. Only `Thuesday-20-02-2018` (sic —
the official misspelling of Tuesday) retains them. Therefore:

- **Arm A** (dataset-level) runs on all ten days.
- **Arm B** (matched flows) runs on Tuesday-20-02 only, and the paper states
  that the official release makes per-flow auditing impossible elsewhere —
  itself a reportable property.

The improved 2018 files are ALREADY in `data/raw_improved` (they shipped in
the same Kaggle download as improved 2017). Only the original needs fetching:
canonical route is the CIC's public S3 bucket (no credentials needed).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = '/content/drive/MyDrive/research/ids-label-correction'
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))

import importlib, config as C, helpers as H
importlib.reload(C); importlib.reload(H)
import pandas as pd, numpy as np, json

for d in ['data/raw_original_2018', 'data/interim']:
    os.makedirs(os.path.join(DRIVE_ROOT, d), exist_ok=True)
RAW18 = os.path.join(DRIVE_ROOT, 'data/raw_original_2018')
print('ready:', RAW18)

Mounted at /content/drive
ready: /content/drive/MyDrive/research/ids-label-correction/data/raw_original_2018


In [2]:
# helpers v3: family rules extended for the 2018 label vocabulary.
# Rewrites src/helpers.py in place (only FAMILY_RULES + infil needle change).
hp = os.path.join(DRIVE_ROOT, 'src', 'helpers.py')
src = open(hp).read()

src = src.replace("""FAMILY_RULES = [
    ('heartbleed', 'Heartbleed'),
    ('web attack', 'WebAttack'),""",
"""FAMILY_RULES = [
    ('heartbleed', 'Heartbleed'),
    ('web attack', 'WebAttack'),
    ('brute force -web', 'WebAttack'),
    ('brute force -xss', 'WebAttack'),
    ('sql injection', 'WebAttack'),""")
# 2018 spells it "Infilteration"; 'infilt' matches both spellings
src = src.replace("('infiltration', '__INFIL__'),", "('infilt', '__INFIL__'),")

open(hp, 'w').write(src)
importlib.reload(H)
test = pd.Series(['Brute Force -Web', 'Brute Force -XSS', 'SQL Injection',
                  'Infilteration', 'FTP-BruteForce', 'SSH-Bruteforce',
                  'DDOS attack-HOIC', 'DoS attacks-Hulk', 'Bot', 'Benign'])
print(pd.DataFrame({'label': test, 'family': H.coarse_class(test)}).to_string())

              label        family
0  Brute Force -Web     WebAttack
1  Brute Force -XSS     WebAttack
2     SQL Injection     WebAttack
3     Infilteration  Infiltration
4    FTP-BruteForce    BruteForce
5    SSH-Bruteforce    BruteForce
6  DDOS attack-HOIC          DDoS
7  DoS attacks-Hulk           DoS
8               Bot           Bot
9            Benign        BENIGN


In [3]:
# Download the ORIGINAL 2018 ML CSVs from the CIC's public S3 bucket (~6.5 GB).
# `aws s3 sync` is idempotent: re-running only fetches what is missing.
!pip -q install awscli

!aws s3 sync --no-sign-request --region us-east-1 "s3://cse-cic-ids2018/Processed Traffic Data for ML Algorithms/" "$RAW18"

files = sorted(f for f in os.listdir(RAW18) if f.endswith('.csv'))
print(len(files), 'files:')
for f in files:
    print(' ', f)
assert any('20-02-2018' in f for f in files), (
    'Thuesday-20-02-2018 (the only day with IPs) is missing - check the sync log')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 49.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
10 files:
  Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
  Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
  Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
  Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
  Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
  Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
  Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
  Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
  Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
  Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv


In [4]:
# FALLBACK - run this cell ONLY if the S3 sync above failed.
# Lists Kaggle mirrors; pick one that contains the per-day CSVs INCLUDING
# "Thuesday-20-02-2018" (the day that retains Flow ID / IPs / ports), then:
#   !kaggle datasets download -d <owner/slug> -p "$RAW18" --unzip
!kaggle datasets list -s "CSE-CIC-IDS2018"


You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [5]:
# Audit: load per-day with 2018 quirk handling, build the census.
# Quirks: header rows repeated inside files (Label=="Label"), the extra
# 4 columns on Thuesday-20, "Infilteration" spelling, stray negative values.
DAY18 = sorted(f for f in os.listdir(RAW18) if f.endswith('.csv'))
print(len(DAY18), 'files')

census, totals = {}, {}
dups_total = 0
for fn in DAY18:
    p = os.path.join(RAW18, fn)
    df = pd.read_csv(p, low_memory=False, encoding='latin-1',
                     dtype=str, usecols=lambda c: True)
    df = H.harmonise(df)
    df = df[df['label'].notna() & (df['label'] != 'Label')]   # embedded headers
    totals[fn] = len(df)
    fam = H.coarse_class(df['label'])
    for k, v in fam.value_counts().items():
        census[k] = census.get(k, 0) + int(v)
    print(f'{fn:55s} {len(df):>10,}')

orig18 = pd.Series(census).sort_values(ascending=False)
print(f'\nTOTAL original 2018 rows (headers removed): {sum(totals.values()):,}')
print(orig18.to_string())
pd.DataFrame({'file': list(totals), 'rows': list(totals.values())}) \
  .to_csv(os.path.join(C.RESULTS, 't18_original_files.csv'), index=False)

10 files
Friday-02-03-2018_TrafficForML_CICFlowMeter.csv          1,048,575
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv          1,048,574
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv          1,048,575
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv        7,948,748
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv          331,100
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv        1,048,575
Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv        1,048,575
Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv       1,048,575
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv       1,048,575
Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv         613,071

TOTAL original 2018 rows (headers removed): 16,232,943
BENIGN          13484708
DDoS             1263933
DoS               654300
BruteForce        380949
Bot               286191
Infiltration      161934
WebAttack            928


In [6]:
# Improved 2018 census (files already in raw_improved, 2018 subtree)
IMP18_FILES = []
for dirpath, _, files in os.walk(C.RAW_IMPROVED):
    for fn in files:
        if fn.lower().endswith('.csv') and '2018' in fn:
            IMP18_FILES.append(os.path.join(dirpath, fn))
print(len(IMP18_FILES), 'improved 2018 files')

census_i, attempted_i, total_i = {}, {}, 0
for p in sorted(IMP18_FILES):
    df = pd.read_csv(p, low_memory=False, usecols=None)
    df = H.harmonise(df)
    total_i += len(df)
    fam = H.coarse_class(df['label'])
    att = H.is_attempted(df['label'])
    for k, v in fam.value_counts().items():
        census_i[k] = census_i.get(k, 0) + int(v)
    for k, v in fam[att].value_counts().items():
        attempted_i[k] = attempted_i.get(k, 0) + int(v)
    print(f'{os.path.basename(p):40s} {len(df):>10,}')
    del df

impr18 = pd.Series(census_i).sort_values(ascending=False)
att18 = pd.Series(attempted_i)
print(f'\nTOTAL improved 2018: {total_i:,}')
tbl = pd.DataFrame({'original': orig18, 'improved': impr18}).fillna(0).astype(int)
tbl['delta'] = tbl['improved'] - tbl['original']
tbl['attempted_in_improved'] = att18.reindex(tbl.index).fillna(0).astype(int)
tbl = tbl.reset_index(names='class')
H.save_table(tbl, 't18_class_census.csv')
tbl

10 improved 2018 files
Friday-02-03-2018.csv                     6,311,371
Friday-16-02-2018.csv                     7,390,266
Friday-23-02-2018.csv                     5,976,481
Thursday-01-03-2018.csv                   6,551,401
Thursday-15-02-2018.csv                   5,410,102
Thursday-22-02-2018.csv                   6,071,153
Tuesday-20-02-2018.csv                    6,054,702
Wednesday-14-02-2018.csv                  5,898,350
Wednesday-21-02-2018.csv                  6,962,593
Wednesday-28-02-2018.csv                  6,568,726

TOTAL improved 2018: 63,195,145
saved /content/drive/MyDrive/research/ids-label-correction/results/t18_class_census.csv (7, 5)


,class,original,improved,delta,attempted_in_improved
0,BENIGN,13484708,59353486,45868778,0
1,Bot,286191,143183,-143008,262
2,BruteForce,380949,393071,12122,298874
3,DDoS,1263933,1374399,110466,251
4,DoS,654300,1840877,1186577,6667
5,Infiltration,161934,89691,-72243,28
6,WebAttack,928,438,-490,155


## What to look for

The 2017 signatures to check for replication: does benign shrink? does one
attack class explode the way Infiltration did? what share of each family is
Attempted? Save the census; notebook 08 builds the matched day.